# Rung 4 — the ML comparison (gradient boosting)

The brief said an ML model is "only worth it if it beats the baseline." So this notebook does one disciplined thing: engineer pre-match features, train a gradient-boosting classifier on the 1X2 outcome, and backtest it against the Dixon–Coles + Elo ensemble on the **same split** with the **same metric** (RPS). Then we keep it only if it wins.

Spoiler, and it's the point of the rung: it doesn't. A well-tuned GBM gets close but loses to the purpose-built model.

Model code: `../src/ml.py`.

In [ ]:
import sys, os, warnings; warnings.filterwarnings("ignore"); sys.path.append(os.path.abspath(".."))
import numpy as np, pandas as pd
from src import data, dixon_coles as dc, elo as E, evaluate, ml

## 1. Data — and a leakage trap worth knowing

Same recent, team-filtered slice as the other rungs. **One subtle bug bit hard here:** pandas' default `sort_values` is *not stable*, so matches sharing a date get scrambled — which silently misaligns the feature rows from their outcomes (we measured 48% alignment, i.e. coin-flip garbage). `build_features` now sorts with `kind="stable"`, and we sort the same way here so the two line up.

In [ ]:
CSV="../data/raw/results.csv"
full = data.load_results(CSV) if os.path.exists(CSV) else data.make_synthetic(16,2000,4)
d = data.filter_teams(data.filter_recent(full, years=8), 10).sort_values("date", kind="stable").reset_index(drop=True)
X, y, dates = ml.build_features(d)
# sanity: features must align with their outcomes
aligned = sum(1 for a,b in zip(y, (evaluate.result_to_outcome(r.home_score,r.away_score) for r in d.itertuples())) if a==b)
print(f"{len(d):,} matches, {X.shape[1]} features | alignment {aligned}/{len(y)} (must be 100%)")
X.head(3)

## 2. The features

All strictly **pre-match** (recorded before each game, updated after) so there's no leakage: the Elo gap, a home flag, rolling goals for/against over the last 5, rolling result form, and games-on-record. Note we use the Elo **gap**, not absolute ratings — absolute Elo drifts over the years and wrecks tree extrapolation on the recent test set (another trap we hit).

In [ ]:
print(list(X.columns))

## 3. Train the gradient booster, backtest vs the ensemble
Same 80/20 time split for everyone.

In [ ]:
cut = int(len(d)*0.8)
clf = ml.fit_classifier(X.iloc[:cut], y[:cut])
ml_p = ml.predict_1x2(clf, X.iloc[cut:])            # [home, draw, away], aligned to d.iloc[cut:]

dm = dc.fit(d.iloc[:cut], xi=0.001); em = E.fit(d.iloc[:cut])
known = set(dm.teams) & set(em.teams)
test = d.iloc[cut:].reset_index(drop=True)
rml, rens, outs = [], [], []
for k, r in test.iterrows():
    if r.home_team in known and r.away_team in known:
        outs.append(evaluate.result_to_outcome(r.home_score, r.away_score))
        rml.append(list(ml_p[k]))
        p = E.ensemble_probs(dm.outcome_probs(r.home_team,r.away_team,neutral=False),
                             em.outcome_probs(r.home_team,r.away_team,neutral=False), 0.6)
        rens.append([p["home"], p["draw"], p["away"]])

br = pd.Series([evaluate.result_to_outcome(h,a) for h,a in zip(d.iloc[:cut].home_score, d.iloc[:cut].away_score)])
base = br.value_counts(normalize=True).reindex(["home","draw","away"]).fillna(0).values
res = pd.DataFrame({
    "model": ["base-rate guess", "GBM (rung 4)", "DC+Elo ensemble"],
    "RPS":   [round(evaluate.mean_scores([base]*len(outs), outs)["rps"],4),
              round(evaluate.mean_scores(rml, outs)["rps"],4),
              round(evaluate.mean_scores(rens, outs)["rps"],4)]})
res

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(5,3))
plt.barh(res.model, res.RPS, color=["#94a3b8","#f59e0b","#34d399"])
for i,v in enumerate(res.RPS): plt.text(v+0.002,i,str(v),va="center",fontsize=9)
plt.gca().invert_yaxis(); plt.xlabel("RPS (lower = better)"); plt.title("Rung 4: ML vs the ensemble"); plt.tight_layout(); plt.show()

## 4. Verdict

- The GBM is genuinely good — well-calibrated, miles better than guessing.
- But it **loses to the Dixon–Coles + Elo ensemble** by a small but consistent margin on RPS.
- So by our own rule, **we don't ship it.** The classical model — Poisson goals + a stable Elo prior — already captures what matters, and a generic learner on the same information can't beat it.

When *might* ML win? With genuinely new information the ensemble can't see: lineups, injuries, xG, travel/rest, player-level data. With only results-derived features, the well-specified statistical model is hard to beat — which is the real lesson of rung 4.

That completes the classical ladder (rungs 1–4). Next, not modelling: a Monte-Carlo tournament simulation for who-wins-it-all odds.